In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [4]:
data_protein = pd.read_csv('resultado/06_regulate_protein.groups.tsv', sep='\t', index_col=0)
data_protein_down = pd.read_csv('resultado/06_downregulate_protein.groups.tsv', sep='\t', index_col=0)
data_protein_up = pd.read_csv('resultado/06_upregulate_protein.groups.tsv', sep='\t', index_col=0)

In [5]:
data_protein.shape, data_protein_down.shape, data_protein_up.shape

((2919, 35), (78, 33), (281, 33))

In [6]:
column_onto = 'Gene Ontology (GO)'
data_protein = data_protein[~data_protein[column_onto].isna()]
data_protein_down = data_protein_down[~data_protein_down[column_onto].isna()]
data_protein_up = data_protein_up[~data_protein_up[column_onto].isna()]

In [7]:
data_protein.shape, data_protein_down.shape, data_protein_up.shape

((2466, 35), (67, 33), (215, 33))

In [8]:
gene_onto = data_protein[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada = [item for sublista in gene_onto.to_list() for item in sublista]
values_count_gene_onto_todos = pd.DataFrame(gene_onto_aplanada).value_counts()

In [9]:
gene_onto_up = data_protein_up[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada_up = [item for sublista in gene_onto_up.to_list() for item in sublista]
values_count_gene_onto_up = pd.DataFrame(gene_onto_aplanada_up).value_counts()

In [10]:
gene_onto_down = data_protein_down[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada_down = [item for sublista in gene_onto_down.to_list() for item in sublista]
values_count_gene_onto_down = pd.DataFrame(gene_onto_aplanada_down).value_counts()

In [11]:
values_count_gene_onto_up
values_count_gene_onto_down
values_count_gene_onto_todos

0                                                             
cytosol [GO:0005829]                                              330
cytoplasm [GO:0005737]                                            303
 metal ion binding [GO:0046872]                                   233
plasma membrane [GO:0005886]                                      228
 ATP binding [GO:0005524]                                         228
                                                                 ... 
 protein localization to outer membrane [GO:0089705]                1
 protein kinase activity [GO:0004672]                               1
 bacterial-type flagellum-dependent cell motility [GO:0071973]      1
 bacteriocin transport [GO:0043213]                                 1
 nucleobase catabolic process [GO:0046113]                          1
Name: count, Length: 2145, dtype: int64

In [12]:
values_count_gene_onto_up.head()

0                                    
plasma membrane [GO:0005886]             31
 ATP binding [GO:0005524]                23
 ATP hydrolysis activity [GO:0016887]    22
 metal ion binding [GO:0046872]          18
cytoplasm [GO:0005737]                   16
Name: count, dtype: int64

In [13]:
N_UP = values_count_gene_onto_up.sum() 
N_DOWN = values_count_gene_onto_down.sum()
N_TOTAL = values_count_gene_onto_todos.sum()

In [14]:
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def calcular_enriquecimiento(counts_target, counts_background, n_target, n_total):
    resultados = []
    
    for term, a in counts_target.items(): # a = genes en mi lista con este GO
        
        total_go_bg = counts_background.get(term, a) 
        b = total_go_bg - a # b = genes en el resto del genoma con este GO
        
        c = n_target - a # c = genes en mi lista SIN este GO
        
        d = (n_total - n_target) - b # d = genes en el resto del genoma SIN este GO
        
        _, p_val = fisher_exact([[a, b], [c, d]], alternative='greater') # Prueba de Fisher (unilateral derecha para "enriquecimiento")
        
        resultados.append({'GO_Term': term, 'Hits': a, 'P_Value': p_val})
    
    df = pd.DataFrame(resultados)
    if not df.empty:
        df['FDR'] = multipletests(df['P_Value'], method='fdr_bh')[1]
    
    return df.sort_values('FDR')


res_up = calcular_enriquecimiento(values_count_gene_onto_up, values_count_gene_onto_todos, N_UP, N_TOTAL)


res_down = calcular_enriquecimiento(values_count_gene_onto_down, values_count_gene_onto_todos, N_DOWN, N_TOTAL)

In [15]:
res_up[res_up['FDR']< 0.05]

,GO_Term,Hits,P_Value,FDR
7,( branched-chain amino acid transmembrane tran...,11,4.845240e-11,1.584394e-08
11,"( L-amino acid transport [GO:0015807],)",7,1.424859e-07,2.329644e-05
2,"( ATP hydrolysis activity [GO:0016887],)",22,5.486373e-05,5.980147e-03
39,"( microcin transport [GO:0042884],)",3,4.962960e-04,2.704813e-02
45,"( glycine cleavage complex [GO:0005960],)",3,4.962960e-04,2.704813e-02
53,( glycine decarboxylation via glycine cleavage...,3,4.962960e-04,2.704813e-02
14,"( transmembrane transport [GO:0055085],)",6,9.741280e-04,3.592443e-02
46,( L-isoleucine import across plasma membrane [...,3,1.867631e-03,3.592443e-02
43,"( L-alanine transport [GO:0015808],)",3,1.867631e-03,3.592443e-02
47,( L-isoleucine transmembrane transporter activ...,3,1.867631e-03,3.592443e-02


In [16]:
res_down[res_down['FDR']< 0.05]

,GO_Term,Hits,P_Value,FDR
4,( reductive pentose-phosphate cycle [GO:001925...,8,8.262696e-13,1.272455e-10
6,"( protein maturation [GO:0051604],)",7,2.711460e-11,2.087824e-09
11,"( ferredoxin hydrogenase activity [GO:0008901],)",4,9.366705e-07,4.808242e-05
8,"( nickel cation binding [GO:0016151],)",4,4.567877e-06,1.758633e-04
1,"( zinc ion binding [GO:0008270],)",10,2.966292e-05,9.136179e-04
34,(carboxyl- or carbamoyltransferase activity [G...,2,9.750436e-04,2.502612e-02
27,"( transketolase activity [GO:0004802],)",2,2.864576e-03,4.901608e-02
37,"( phosphoglycerate kinase activity [GO:0004618],)",2,2.864576e-03,4.901608e-02
10,"( glycolytic process [GO:0006096],)",4,2.503026e-03,4.901608e-02


In [17]:
res_up[res_up['FDR']< 0.05].to_csv('resultado/go_term_upregulate.csv')
res_down[res_down['FDR']< 0.05].to_csv('resultado/go_term_downregulate.csv')

In [18]:
res_up.columns = ['GO_Term', 'Hits_up', 'P_Value_up', 'FDR_up']

In [19]:
res_down

,GO_Term,Hits,P_Value,FDR
4,( reductive pentose-phosphate cycle [GO:001925...,8,8.262696e-13,1.272455e-10
6,"( protein maturation [GO:0051604],)",7,2.711460e-11,2.087824e-09
11,"( ferredoxin hydrogenase activity [GO:0008901],)",4,9.366705e-07,4.808242e-05
8,"( nickel cation binding [GO:0016151],)",4,4.567877e-06,1.758633e-04
1,"( zinc ion binding [GO:0008270],)",10,2.966292e-05,9.136179e-04
...,...,...,...,...
137,"( hydrolase activity [GO:0016787],)",1,7.376911e-01,7.573629e-01
121,( flavin adenine dinucleotide binding [GO:0050...,1,7.539274e-01,7.689061e-01
57,"( sequence-specific DNA binding [GO:0043565],)",1,7.616661e-01,7.716880e-01
5,"(cytosol [GO:0005829],)",7,8.974240e-01,9.032895e-01


In [20]:
res_down.columns = ['GO_Term', 'Hits_down', 'P_Value_down', 'FDR_down']

In [21]:
pd.merge(res_up['GO_Term'], res_down['GO_Term'])

,GO_Term
0,"( ATP hydrolysis activity [GO:0016887],)"
1,"( propionate metabolic process, methylcitrate ..."
2,"(plasma membrane [GO:0005886],)"
3,"( molybdopterin cofactor binding [GO:0043546],)"
4,"(oxidoreductase activity [GO:0016491],)"
...,...
64,( phosphorelay sensor kinase activity [GO:0000...
65,"( methylation [GO:0032259],)"
66,"(cytoplasm [GO:0005737],)"
67,"(cytosol [GO:0005829],)"


In [22]:
df_comparativo = pd.merge(
    res_up, 
    res_down, 
    on='GO_Term', 
    how='outer', 
)